### Instruct Models & the Chat Template

In the Text Generation demo we prompted a **base** model — it just continues whatever text you give it. This demo is about **instruct** (a.k.a. **chat**) models: the same kind of transformer, taken one step further so that it *follows instructions* and *holds a conversation* instead of merely autocompleting.

Two ideas do all the work here:

1. **Instruct models.** A base model is trained only to predict the next token over raw text. An instruct model is that base model fine-tuned on examples of following instructions and chatting (supervised fine-tuning, then preference tuning). Uses `.generate()` — it just *behaves* the way people expect from a chatbot.
2. **The chat template.** Chat models aren't fed a plain string; they're fed a *conversation* as a list of `{"role", "content"}` messages, flattened into the exact format the model was trained on. `apply_chat_template` does that flattening for you, so you never hand-write a model's special tokens.

Get those two right and you can make a small model summarize, answer questions, adopt a persona, and carry a multi-turn chat — all through one clean pattern.

#### Setup

We're on Transformers v5 (PyTorch-only). A GPU runtime is nice but not required — this model is tiny enough to run on CPU.

In [2]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

Using device: cuda


#### The model: `HuggingFaceTB/SmolLM2-360M-Instruct`

https://huggingface.co/HuggingFaceTB/SmolLM2-360M-Instruct

We're using a deliberately **tiny** instruct model (360M parameters) from HuggingFaceTB — Hugging Face's own research team. Two practical reasons:

- It's **small** (~720 MB), so it downloads in seconds and runs anywhere. Perfect for a live session.
- It's **ungated** — no license to accept, no access token needed.

At 360M parameters this model is for *learning the mechanics*, not for impressive answers. It will sometimes be repetitive, shaky, or plain wrong — that's expected for a sub-1B model

In [3]:
checkpoint = "HuggingFaceTB/SmolLM2-360M-Instruct"

tokenizer = AutoTokenizer.from_pretrained(checkpoint)
model = AutoModelForCausalLM.from_pretrained(checkpoint).to(device)

model

config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.76k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/801k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.10M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/724M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(49152, 960, padding_idx=2)
    (layers): ModuleList(
      (0-31): 32 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=960, out_features=960, bias=False)
          (k_proj): Linear(in_features=960, out_features=320, bias=False)
          (v_proj): Linear(in_features=960, out_features=320, bias=False)
          (o_proj): Linear(in_features=960, out_features=960, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=960, out_features=2560, bias=False)
          (up_proj): Linear(in_features=960, out_features=2560, bias=False)
          (down_proj): Linear(in_features=2560, out_features=960, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((960,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((960,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((960,), eps=1e-05)
    (r

#### The messages format

A conversation is a list of dictionaries, each with a **role** and its **content**. Three roles cover almost everything:

- `system` — sets the assistant's overall behaviour/persona (optional).
- `user` — what the person says.
- `assistant` — what the model says back.

In [4]:
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "What is the capital of France?"},
]

messages

[{'role': 'system', 'content': 'You are a helpful assistant.'},
 {'role': 'user', 'content': 'What is the capital of France?'}]

#### `apply_chat_template`

The model doesn't actually consume that list of dictionaries — under the hood it still reads one flat string of tokens. Each chat model ships a **chat template** (a small Jinja template inside its tokenizer) that describes exactly how to flatten the messages into the string it was trained on, including the special delimiter tokens that mark where each turn begins and ends.

tokenize=False` let's us *see* that string instead of token IDs. We also pass `add_generation_prompt=True`, which appends the tokens that say "the assistant speaks next" — the cue that makes the model *reply* rather than continue the user's sentence.

In [5]:
prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)

print(prompt)

<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
What is the capital of France?<|im_end|>
<|im_start|>assistant



Those `<|im_start|>` / `<|im_end|>` markers are **ChatML**, the template family SmolLM2 (and Qwen) use. Notice how each turn is wrapped, and how the string ends with `<|im_start|>assistant` — that trailing opener is what `add_generation_prompt=True` added, telling the model it's now the assistant's turn to speak.

A different model (Llama, Mistral, Gemma) uses completely different delimiters, but `apply_chat_template` reads each model's own template, so the *same* messages list works everywhere.

#### Generating a reply


In [8]:
inputs = tokenizer(prompt, return_tensors="pt").to(device)

outputs = model.generate(
    **inputs,
    max_new_tokens=100
)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

system
You are a helpful assistant.
user
What is the capital of France?
assistant
The capital of France is Paris.


Notice the decoded output contains the **whole** conversation, including our prompt — because `generate()` returns the input tokens followed by the new ones. Usually we only want the assistant's fresh reply, so we slice off the input tokens (everything up to the length of `inputs`) before decoding.

In [10]:
reply_ids = outputs[0][inputs["input_ids"].shape[-1]:]

print(reply_ids)
print(tokenizer.decode(reply_ids, skip_special_tokens=True))

tensor([ 504, 3575,  282, 4649,  314, 7042,   30,    2], device='cuda:0')
The capital of France is Paris.


#### Wrapping it in a helper



In [11]:
def chat(messages, max_new_tokens=120):
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
    )

    reply_ids = outputs[0][inputs["input_ids"].shape[-1]:]
    return tokenizer.decode(reply_ids, skip_special_tokens=True)

In [12]:
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "Give me three ideas for a weekend project."},
]

print(chat(messages))

1. DIY Woodworking Project: Create a beautiful wooden birdhouse or a small tabletop planter. You can use reclaimed wood, screws, and nails to make it.

2. DIY Garden Project: Create a garden with a variety of flowers, herbs, and vegetables. You can use seeds, seedlings, and gardening tools to make it.

3. DIY Water Feature: Build a small water feature like a birdbath, a small fountain, or a small pond. You can use materials like plastic bottles, metal, or wood to make it.


#### Steering behaviour with the system prompt

In [13]:
messages = [
    {"role": "system", "content": "You are a helpful assistant. Always answer in exactly one short sentence."},
    {"role": "user", "content": "What is a computer?"},
]

print(chat(messages))

A computer is an electronic device that can perform calculations and store information.


#### Multi-turn conversation

This is *why* the messages format exists. To continue a conversation, you append the assistant's reply to the list and add the next user turn, then send the whole history back. The model sees the full context, so follow-ups like "and another one" make sense.

In [14]:
conversation = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "Suggest a name for a pet cat."},
]

# First reply
reply = chat(conversation)
print("Assistant:", reply)

# Append the reply, then ask a follow-up that relies on context
conversation.append({"role": "assistant", "content": reply})
conversation.append({"role": "user", "content": "Now suggest one for a dog."})

print("Assistant:", chat(conversation))

Assistant: A name for a pet cat could be Whisker, Whisker, Whisker, Whisker, Whisker, Whisker, Whisker, Whisker, Whisker, Whisker, Whisker, Whisker, Whisker, Whisker, Whisker, Whisker, Whisker, Whisker, Whisker, Whisker, Whisker, Whisker, Whisker, Whisker, Whisker, Whisker, Whisker, Whisker,
Assistant: A name for a dog could be Binky, Binky, Binky, Binky, Binky, Binky, Binky, Binky, Binky, Binky, Binky, Binky, Binky, Binky, Binky, Binky, Binky, Binky, Binky, Binky, Binky, Binky, Binky, Binky, Binky, Binky, Binky, Binky, B


#### The pipeline shortcut

Once you understand what's happening, the `text-generation` pipeline offers a one-liner: hand it the **messages list directly** and it applies the chat template for you internally. The reply comes back as the conversation with the assistant's new message appended, so we grab the last message's content.

In [15]:
from transformers import pipeline

chatbot = pipeline(
    "text-generation",
    model=checkpoint,
    device=0 if torch.cuda.is_available() else -1,
)

messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "What is the capital of France?"},
]

output = chatbot(messages, max_new_tokens=100)

print(output[0]["generated_text"][-1]["content"])

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPT2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


The capital of France is Paris.


*(Tip: depending on your Transformers version, passing generation settings straight to the pipeline may print a small note about generation parameters. It's harmless — see the Text Generation demo for the tidy `GenerationConfig` way to pass them.)*

#### Classic tasks, done by a chat model

In Transformers v5 the dedicated `summarization`, `translation`, and `question-answering` pipelines were removed — the recommended way to do those now is to **ask a chat model**. With the pattern above, that's just a well-worded user message.

In [16]:
passage = (
    "The Hugging Face Hub hosts hundreds of thousands of models, datasets, and demos. "
    "It lets people download pretrained models, share their own, and try them in the browser. "
    "Its libraries, like Transformers and Datasets, have become standard tools for building NLP applications."
)

messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": f"Summarize the following in one sentence:\n\n{passage}"},
]

print(chat(messages))

The Hugging Face Hub hosts a vast collection of pretrained models, datasets, and demos, allowing users to download, share, and use them in web applications.
